In [172]:
import pandas as pd
import numpy as np

# Import data

## 01. Pollutants

In [173]:
pollutant_files = [
    "data/air_pollutants/ugz_ogd_air_d1_2017.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2018.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2019.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2020.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2021.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2022.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2023.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2024.csv"
]

to_merge = []
for file in pollutant_files:
    df_temp = pd.read_csv(file)
    to_merge.append(df_temp)

df_pollutants = pd.concat(to_merge, ignore_index = True)

df_pollutants.head()

,Datum,Standort,Parameter,Intervall,Einheit,Wert,Status
0,2017-01-01T00:00+0100,Zch_Heubeeribüel,NO2,d1,µg/m3,32.78,bereinigt
1,2017-01-01T00:00+0100,Zch_Heubeeribüel,NO,d1,µg/m3,10.27,bereinigt
2,2017-01-01T00:00+0100,Zch_Heubeeribüel,NOx,d1,ppb,25.38,bereinigt
3,2017-01-01T00:00+0100,Zch_Heubeeribüel,O3,d1,µg/m3,6.44,bereinigt
4,2017-01-01T00:00+0100,Zch_Heubeeribüel,O3_max_h1,d1,µg/m3,17.86,bereinigt


In [174]:
# Make new columns with Day and Time from Datum
Datum_split = df_pollutants["Datum"].str.split("T", n=1, expand=True)
df_pollutants["Day"] = Datum_split[0]
df_pollutants["Time"] = Datum_split[1]

df_pollutants.head()

,Datum,Standort,Parameter,Intervall,Einheit,Wert,Status,Day,Time
0,2017-01-01T00:00+0100,Zch_Heubeeribüel,NO2,d1,µg/m3,32.78,bereinigt,2017-01-01,00:00+0100
1,2017-01-01T00:00+0100,Zch_Heubeeribüel,NO,d1,µg/m3,10.27,bereinigt,2017-01-01,00:00+0100
2,2017-01-01T00:00+0100,Zch_Heubeeribüel,NOx,d1,ppb,25.38,bereinigt,2017-01-01,00:00+0100
3,2017-01-01T00:00+0100,Zch_Heubeeribüel,O3,d1,µg/m3,6.44,bereinigt,2017-01-01,00:00+0100
4,2017-01-01T00:00+0100,Zch_Heubeeribüel,O3_max_h1,d1,µg/m3,17.86,bereinigt,2017-01-01,00:00+0100


In [175]:
# Check the measurements are always only once a day
df_pollutants["Time"].unique()

array(['00:00+0100'], dtype=object)

In [176]:
# Drop columns we don't need
df_pollutants.drop(columns=["Datum", "Intervall", "Status", "Time"], inplace=True)

# Rename the columns
df_pollutants.rename(columns={
    "Standort": "Location",
    "Einheit": "Pollutants_unit",
    "Wert": "Pollutants_value",
    "Parameter": "Pollutants_parameter"
}, inplace=True)

# Re-arrange the columns to have day as first
df_pollutants = df_pollutants[["Day", "Location", "Pollutants_parameter", "Pollutants_value", "Pollutants_unit"]]
df_pollutants.head()

df_pollutants.to_csv("data/Processed_daily/pollutants_dataset_daily.csv", index=False)


## 02. Meteo

In [177]:
# Meteo data
meteo_files = [
    "data/ugz_meteo/ugz_ogd_meteo_d1_2017.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2018.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2019.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2020.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2021.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2022.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2023.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2024.csv"
]

# Delete all quotes from all files to fix the double-quoted values and make a new file
cleaned_lines = []
for file in meteo_files:
    with open(file, "r") as f:
        lines = f.readlines()
        cleaned_lines.append([line.replace('"', '') for line in lines])

with open("data/ugz_meteo/ugz_ogd_meteo_d1_2017-2024_cleaned.csv", "w") as f:
    for line in cleaned_lines:
        f.writelines(line)

# Create dataframe from new file without quotes
df_meteo = pd.read_csv("data/ugz_meteo/ugz_ogd_meteo_d1_2017-2024_cleaned.csv")
df_meteo.head()

,Datum,Standort,Parameter,Intervall,Einheit,Wert,Status
0,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,RainDur,d1,min,0,bereinigt
1,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,T,d1,°C,-1.51,bereinigt
2,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,p,d1,hPa,972.88,bereinigt
3,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,T_max_h1,d1,°C,1.6,bereinigt
4,2017-01-01T00:00+0100,Zch_Schimmelstrasse,RainDur,d1,min,0,bereinigt


In [178]:
# Make new columns with Day and Time from Datum
Datum_split = df_meteo["Datum"].str.split("T", n=1, expand=True)
df_meteo["Day"] = Datum_split[0]
df_meteo["Time"] = Datum_split[1]

df_meteo.head()

,Datum,Standort,Parameter,Intervall,Einheit,Wert,Status,Day,Time
0,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,RainDur,d1,min,0,bereinigt,2017-01-01,00:00+0100
1,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,T,d1,°C,-1.51,bereinigt,2017-01-01,00:00+0100
2,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,p,d1,hPa,972.88,bereinigt,2017-01-01,00:00+0100
3,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,T_max_h1,d1,°C,1.6,bereinigt,2017-01-01,00:00+0100
4,2017-01-01T00:00+0100,Zch_Schimmelstrasse,RainDur,d1,min,0,bereinigt,2017-01-01,00:00+0100


In [179]:
# Check the measurements are always only once a day
df_meteo["Time"].unique()

array(['00:00+0100', None], dtype=object)

In [180]:
# Drop columns we don't need
df_meteo.drop(columns=["Datum", "Intervall", "Status", "Time"], inplace=True)

# Rename the columns
df_meteo.rename(columns={
    "Standort": "Location",
    "Einheit": "Meteo_unit",
    "Wert": "Meteo_value",
    "Parameter": "Meteo_parameter"
}, inplace=True)

# Re-arrange the columns to have day as first
df_meteo = df_meteo[["Day", "Location", "Meteo_parameter", "Meteo_value", "Meteo_unit"]]
df_meteo.head()

df_meteo.to_csv("data/Processed_daily/meteo_dataset_daily.csv", index=False)

## 03. Traffic

In [181]:
# Create dataframe from the traffic file
df_traffic = pd.read_csv("data/traffic_data_17_24.csv")
df_traffic.head()

,ZSName,MessungDatZeit,AnzFahrzeuge,Year,Datetime,pollutant_station
0,Rosengartenstrasse (Nordstrasse),2017-01-01T00:00:00,611.0,2017,2017-01-01 00:00:00,Zch_Rosengartenstrasse
1,Rosengartenstrasse (Nordstrasse),2017-01-01T01:00:00,1278.0,2017,2017-01-01 01:00:00,Zch_Rosengartenstrasse
2,Rosengartenstrasse (Nordstrasse),2017-01-01T02:00:00,950.0,2017,2017-01-01 02:00:00,Zch_Rosengartenstrasse
3,Rosengartenstrasse (Nordstrasse),2017-01-01T03:00:00,716.0,2017,2017-01-01 03:00:00,Zch_Rosengartenstrasse
4,Rosengartenstrasse (Nordstrasse),2017-01-01T04:00:00,638.0,2017,2017-01-01 04:00:00,Zch_Rosengartenstrasse


In [182]:
# Make new columns with Day and Time from Datum
Datum_split = df_traffic["MessungDatZeit"].str.split("T", n=1, expand=True)
df_traffic["Day"] = Datum_split[0]
df_traffic["Time"] = Datum_split[1]

df_traffic.head()

,ZSName,MessungDatZeit,AnzFahrzeuge,Year,Datetime,pollutant_station,Day,Time
0,Rosengartenstrasse (Nordstrasse),2017-01-01T00:00:00,611.0,2017,2017-01-01 00:00:00,Zch_Rosengartenstrasse,2017-01-01,00:00:00
1,Rosengartenstrasse (Nordstrasse),2017-01-01T01:00:00,1278.0,2017,2017-01-01 01:00:00,Zch_Rosengartenstrasse,2017-01-01,01:00:00
2,Rosengartenstrasse (Nordstrasse),2017-01-01T02:00:00,950.0,2017,2017-01-01 02:00:00,Zch_Rosengartenstrasse,2017-01-01,02:00:00
3,Rosengartenstrasse (Nordstrasse),2017-01-01T03:00:00,716.0,2017,2017-01-01 03:00:00,Zch_Rosengartenstrasse,2017-01-01,03:00:00
4,Rosengartenstrasse (Nordstrasse),2017-01-01T04:00:00,638.0,2017,2017-01-01 04:00:00,Zch_Rosengartenstrasse,2017-01-01,04:00:00


In [183]:
# Drop columns we don't need
df_traffic.drop(columns=["ZSName", "MessungDatZeit", "Year", "Datetime", "Time"], inplace=True)

# Rename the columns
df_traffic.rename(columns={
    "AnzFahrzeuge": "Number_of_vehicles",
    "pollutant_station": "Location"
}, inplace=True)

# Re-arrange the columns to have day as first
df_traffic = df_traffic[["Day", "Location", "Number_of_vehicles"]]
df_traffic.head()

,Day,Location,Number_of_vehicles
0,2017-01-01,Zch_Rosengartenstrasse,611.0
1,2017-01-01,Zch_Rosengartenstrasse,1278.0
2,2017-01-01,Zch_Rosengartenstrasse,950.0
3,2017-01-01,Zch_Rosengartenstrasse,716.0
4,2017-01-01,Zch_Rosengartenstrasse,638.0


In [184]:
# TODO: There are "null" values in the traffic, does that mean 0 or NA?
# Make sum of Number_of_vehicles per each day and location
df_traffic = (
    df_traffic.groupby(["Day", "Location"], as_index=False)["Number_of_vehicles"]
    .sum()
)

df_traffic.head()
df_traffic.to_csv("data/Processed_daily/traffic_dataset_daily.csv", index=False)


## 04. Combined dataset

In [185]:
# Check that locations are in all datasets
set_pollutants = set(df_pollutants["Location"].unique())
set_meteo = set(df_meteo["Location"].unique())
set_traffic = set(df_traffic["Location"].unique())

combined_set = set_pollutants & set_meteo & set_traffic

print(f"Found in all three: {combined_set}")
print(f"Extra in Pollutants: {set_pollutants - combined_set or "nothing"}")
print(f"Extra in Meteo: {set_meteo - combined_set or "nothing"}")
print(f"Extra in Traffic: {set_traffic - combined_set or "nothing"}")

Found in all three: {'Zch_Heubeeribüel', 'Zch_Rosengartenstrasse', 'Zch_Schimmelstrasse', 'Zch_Stampfenbachstrasse'}
Extra in Pollutants: nothing
Extra in Meteo: {'Standort'}
Extra in Traffic: nothing


In [186]:
# Check if there are any unique days in any dataset
set_pollutants = set(df_pollutants["Day"].unique())
set_meteo = set(df_meteo["Day"].unique())
set_traffic = set(df_traffic["Day"].unique())

combined_set = set_pollutants & set_meteo & set_traffic

print(f"Extra in Pollutants: {set_pollutants - combined_set or "nothing"}")
print(f"Extra in Meteo: {set_meteo - combined_set or "nothing"}")
print(f"Extra in Traffic: {set_traffic - combined_set or "nothing"}")

Extra in Pollutants: nothing
Extra in Meteo: {'\ufeffDatum'}
Extra in Traffic: nothing


In [187]:
# Drop rows with "Standort" in "Location" column in Meteo dataframe
df_meteo.drop(df_meteo[df_meteo["Location"] == "Standort"].index, inplace=True)

# Combine all datasets across days and locations
df_combined = df_pollutants.merge(df_meteo, on=["Day", "Location"], how="outer") \
    .merge(df_traffic, on=["Day", "Location"], how="outer")

df_combined.head()
df_combined.to_csv("data/Processed_daily/combined_dataset_daily.csv", index=False)

In [188]:
# Check NaN values per column
df_combined.isna().sum()

Day                         0
Location                    0
Pollutants_parameter        0
Pollutants_value         7324
Pollutants_unit             0
Meteo_parameter         15303
Meteo_value             20048
Meteo_unit              15303
Number_of_vehicles      16743
dtype: int64

In [196]:
#Check NAN values per row
print(f"Total rows: {len(df_combined)}")
#Check how many rows have some missing values
df_combined.isna().sum(axis=1)[df_combined.isna().sum(axis=1) != 0]


Total rows: 381840


0         4
1         4
2         4
3         4
4         4
         ..
379698    1
379699    1
379700    1
379701    1
379702    1
Length: 25350, dtype: int64